In [ ]:
!pip install -q streamlit pyngrok scikit-learn pandas numpy joblib

In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:
import pandas as pd

df = pd.read_csv("Datasetprojpowerbi.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nFirst 5 rows:")
display(df.head())

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDuplicate reports:", df["Reports"].duplicated().sum())

print("\nEmpty reports:", df["Reports"].isna().sum())

In [ ]:
df = df.dropna(subset=["Reports", "Genre"])

df["Reports"] = df["Reports"].astype(str).str.strip()
df["Genre"] = df["Genre"].astype(str).str.strip()

df = df[df["Reports"] != ""]

df = df.drop_duplicates(subset=["Reports"])

df = df.reset_index(drop=True)

print("Clean dataset shape:", df.shape)

In [ ]:
print(df["Genre"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

X = df["Reports"]
y = df["Genre"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

category_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=2000
    ))
])

category_model.fit(X_train, y_train)

predictions = category_model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(
    "Category Classification Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0
    )
)

In [ ]:
test_issues = [
    "I am unable to access the online learning platform",
    "The hostel room has a water problem",
    "I need help regarding my career opportunities",
    "The canteen food quality is very poor",
    "I need financial assistance for my education"
]

for issue in test_issues:
    prediction = category_model.predict([issue])[0]

    print("Issue:", issue)
    print("Predicted Category:", prediction)
    print("-" * 70)

In [ ]:
import sys
import os

test_cases = [
    "The projector in Block A classroom is not working.",
    "The food served in the college canteen is very poor.",
    "I am unable to access the online learning platform.",
    "I need financial assistance for my education fees.",
    "There is a fire emergency in the hostel and students are in danger.",
    "I need guidance regarding career opportunities.",
    "There is a water problem in the hostel.",
    "I have difficulty accessing academic resources."
]

# Safely verify that the function exists in the current notebook memory, or import from app.py
if 'analyze_grievance' not in globals():
    try:
        current_dir = os.path.abspath(os.getcwd())
        if current_dir not in sys.path:
            sys.path.insert(0, current_dir)
        from app import analyze_grievance
    except ImportError:
        raise NameError(
            "The function 'analyze_grievance' is not defined in this session and app.py could not be found. "
            "Please run the cells below (such as cell HdNybsMKaqNo) to define the function first."
        )

print("GRIEVAI RESULTS")
print("=" * 80)

for i, issue in enumerate(test_cases, 1):
    result = analyze_grievance(issue)

    # Handle both dictionary and tuple return formats dynamically
    if isinstance(result, dict):
        category = result["category"]
        priority = result["priority"]
        score = result["score"]
        department = result["department"]
        action = result["action"]
        reasons = result["reasons"]
    else:
        category, priority, score, department, action, reasons = result

    print("\nTest Case:", i)
    print("Grievance:", issue)
    print("Category:", category)
    print("Priority:", priority)
    print("Priority Score:", score, "/ 10")
    print("Department:", department)
    print("Recommended Action:", action)
    if isinstance(reasons, list):
        print("Reason:", ", ".join(reasons))
    else:
        print("Reason:", reasons)
    print("-" * 80)

In [ ]:
# First, let's analyze an issue to populate these variables
category, priority, score, department, action, reasons = analyze_grievance("The projector in Block A classroom is not working.")

# Now we can print them directly
print("Category:", category)
print("Priority:", priority)
print("Score:", score)

In [ ]:

def calculate_priority(text):
    text = text.lower()

    critical_words = [
        "emergency",
        "fire",
        "accident",
        "danger",
        "unsafe",
        "violence",
        "medical emergency"
    ]

    high_words = [
        "urgent",
        "not working",
        "broken",
        "no water",
        "water problem",
        "water supply",
        "water shortage",
        "water issue",
        "water is not coming",
        "water not coming",
        "water not working",
        "water is not working",
        "water unavailable",
        "water not available",
        "power failure",
        "electricity",
        "security",
        "food poisoning",
        "severe",
        "stopped working"
    ]

    medium_words = [
        "delay",
        "problem",
        "issue",
        "difficulty",
        "unable",
        "poor"
    ]

    low_words = [
        "information",
        "query",
        "suggestion",
        "feedback",
        "enquiry",
        "career guidance",
        "guidance"
    ]

    for word in critical_words:
        if word in text:
            return "Critical", 10, "Critical safety or emergency indicator"

    for word in high_words:
        if word in text:
            return "High", 4, "High urgency or service disruption"

    for word in medium_words:
        if word in text:
            return "Medium", 2, "General service issue"

    for word in low_words:
        if word in text:
            return "Low", 1, "Information or feedback request"

    return "Low", 0, "No strong urgency indicators detected"


department_map = {
    "Academic Support and Resources": "Academic Support Cell",
    "Food and Cantines": "Canteen and Food Services",
    "Financial Support": "Student Finance Cell",
    "Online learning": "Digital Learning Cell",
    "Career opportunities": "Career Development Cell",
    "International student experiences": "International Student Cell",
    "Athletics and sports": "Sports Department",
    "Housing and Transportation": "Hostel and Transport Cell",
    "Health and Well-being Support": "Health and Wellness Cell",
    "Activities and Travelling": "Student Activities Cell",
    "Student Affairs": "Student Affairs Cell",
    "Emergency and Safety": "Emergency and Safety Cell"
}


action_map = {
    "Academic Support and Resources": "Assign the grievance to academic or technical support staff.",
    "Food and Cantines": "Forward the grievance to the canteen and food services team.",
    "Financial Support": "Forward the grievance to the student finance cell.",
    "Online learning": "Assign the issue to the digital learning support team.",
    "Career opportunities": "Forward the request to the career development cell.",
    "International student experiences": "Forward the grievance to the international student support cell.",
    "Athletics and sports": "Forward the issue to the sports department.",
    "Housing and Transportation": "Forward the grievance to the hostel or transport team.",
    "Health and Well-being Support": "Forward the grievance to the health and wellness cell.",
    "Activities and Travelling": "Forward the request to the student activities cell.",
    "Student Affairs": "Forward the grievance to the student affairs cell.",
    "Emergency and Safety": "Immediately forward the grievance to the appropriate emergency and safety response team."
}


def analyze_grievance(issue):
    text = issue.lower()

    emergency_words = [
        "fire",
        "emergency",
        "accident",
        "danger",
        "unsafe",
        "violence",
        "medical emergency"
    ]

    emergency_found = any(
        word in text for word in emergency_words
    )

    priority, score, reasons = calculate_priority(issue)

    if emergency_found:
        category = "Emergency and Safety"

    elif "water" in text:
        category = "Housing and Transportation"

        if score < 4:
            priority = "High"
            score = 4
            reasons = "High urgency or service disruption"

    elif (
        "projector" in text
        or "classroom" in text
        or "not working" in text
        or "broken" in text
    ):
        category = "Academic Support and Resources"

    else:
        category = category_model.predict([issue])[0]

    department = department_map.get(
        category,
        "Student Grievance Cell"
    )

    action = action_map.get(
        category,
        "Forward the grievance to the Student Grievance Cell."
    )

    return category, priority, score, department, action, reasons


In [ ]:
test_cases = [
    "The projector in Block A classroom is not working.",
    "The food served in the college canteen is very poor.",
    "I am unable to access the online learning platform.",
    "I need financial assistance for my education fees.",
    "There is a fire emergency in the hostel and students are in danger.",
    "I need guidance regarding career opportunities.",
    "There is a water problem in the hostel.",
    "I have difficulty accessing academic resources."
]

print("GRIEVAI RESULTS")
print("=" * 80)

for i, issue in enumerate(test_cases, 1):

    category, priority, score, department, action, reasons = analyze_grievance(issue)

    print("\nTest Case:", i)
    print("Grievance:", issue)
    print("Category:", category)
    print("Priority:", priority)
    print("Priority Score:", score, "/ 10")
    print("Department:", department)
    print("Recommended Action:", action)
    print("Reason:", ", ".join(reasons))
    print("-" * 80)

In [ ]:
def calculate_priority(text):

    text = text.lower()

    score = 0
    reasons = []

    critical_words = [
        "emergency",
        "fire",
        "accident",
        "danger",
        "unsafe",
        "violence",
        "medical emergency"
    ]

    high_words = [
        "urgent",
        "not working",
        "broken",
        "no water",
        "water problem",
        "power failure",
        "electricity",
        "security",
        "food poisoning",
        "severe",
        "stopped working"
    ]

    medium_words = [
        "delay",
        "problem",
        "issue",
        "difficulty",
        "unable",
        "poor"
    ]

    low_words = [
        "information",
        "query",
        "suggestion",
        "feedback",
        "enquiry",
        "career guidance"
    ]

    for word in critical_words:
        if word in text:
            score += 5
            reasons.append("Critical safety or emergency indicator")

    for word in high_words:
        if word in text:
            score += 4
            reasons.append("High urgency or service disruption")

    for word in medium_words:
        if word in text:
            score += 1
            reasons.append("General service issue")

    for word in low_words:
        if word in text:
            score += 1
            reasons.append("Information or feedback request")

    if score >= 8:
        priority = "Critical"
    elif score >= 4:
        priority = "High"
    elif score >= 2:
        priority = "Medium"
    else:
        priority = "Low"

    if score > 10:
        score = 10

    if not reasons:
        reasons.append("No strong urgency indicators detected")

    return priority, score, reasons

In [ ]:
result = analyze_grievance(
    "the water in the a block is not working"
)

print("Category:", result[0])
print("Priority:", result[1])
print("Priority Score:", result[2], "/ 10")
print("Responsible Department:", result[3])
print("Recommended Action:", result[4])
print("Priority Reason:", result[5])

In [ ]:
department_map = {
    "Academic Support and Resources": "Academic Support Cell",
    "Food and Cantines": "Canteen and Food Services",
    "Financial Support": "Student Finance Cell",
    "Online learning": "Digital Learning Cell",
    "Career opportunities": "Career Development Cell",
    "International student experiences": "International Student Cell",
    "Athletics and sports": "Sports Department",
    "Housing and Transportation": "Hostel and Transport Cell",
    "Health and Well-being Support": "Health and Wellness Cell",
    "Activities and Travelling": "Student Activities Cell",
    "Student Affairs": "Student Affairs Cell"
}

In [ ]:
action_map = {
    "Academic Support and Resources": "Assign the grievance to academic or technical support staff.",
    "Food and Cantines": "Forward the grievance to the canteen and food services team.",
    "Financial Support": "Forward the grievance to the student finance cell.",
    "Online learning": "Assign the issue to the digital learning support team.",
    "Career opportunities": "Forward the request to the career development cell.",
    "International student experiences": "Forward the grievance to the international student support cell.",
    "Athletics and sports": "Forward the issue to the sports department.",
    "Housing and Transportation": "Forward the grievance to the hostel or transport team.",
    "Health and Well-being Support": "Forward the grievance to the health and wellness cell.",
    "Activities and Travelling": "Forward the request to the student activities cell.",
    "Student Affairs": "Forward the grievance to the student affairs cell."
}

In [ ]:
def analyze_grievance(issue):

    text = issue.lower()

    emergency_words = [
        "fire",
        "emergency",
        "accident",
        "danger",
        "unsafe",
        "violence",
        "medical emergency"
    ]

    emergency_found = False

    for word in emergency_words:
        if word in text:
            emergency_found = True

    priority, score, reasons = calculate_priority(issue)

    if emergency_found:
        category = "Emergency and Safety"
    else:
        category = category_model.predict([issue])[0]

    department = department_map.get(
        category,
        "Student Grievance Cell"
    )

    action = action_map.get(
        category,
        "Forward the grievance to the Student Grievance Cell."
    )

    return category, priority, score, department, action, reasons

In [ ]:

def analyze_grievance(issue):
    text = issue.lower()

    emergency_words = [
        "fire", "emergency", "accident", "danger",
        "unsafe", "violence", "medical emergency"
    ]

    water_words = [
        "water is not coming", "no water",
        "water supply", "water problem",
        "water shortage", "water not coming",
        "no water supply", "water stopped"
    ]

    emergency_found = any(word in text for word in emergency_words)
    water_found = any(word in text for word in water_words)

    priority, score, reasons = calculate_priority(issue)

    if emergency_found:
        category = "Emergency and Safety"
    elif water_found:
        category = "Housing and Transportation"
        priority = "High"
        score = 4
        reasons = "High urgency or service disruption"
    else:
        category = category_model.predict([issue])[0]

    department = department_map.get(
        category,
        "Student Grievance Cell"
    )

    action = action_map.get(
        category,
        "Forward the grievance to the Student Grievance Cell."
    )

    return category, priority, score, department, action, reasons

In [ ]:
department_map["Emergency and Safety"] = "Emergency and Safety Cell"

action_map["Emergency and Safety"] = "Immediately forward the grievance to the appropriate emergency and safety response team."

In [ ]:
issue = "The projector in Block A classroom is not working"

# Unpack the returned tuple from analyze_grievance directly into variables
category, priority, score, department, action, reasons = analyze_grievance(issue)

print("GRIEVAI RESULT")
print("=" * 50)

print("Grievance:", issue)
print("Category:", category)
print("Priority:", priority)
print("Priority Score:", score, "/ 10")
print("Department:", department)
print("Recommended Action:", action)
print("Reason:", ", ".join(reasons))

In [ ]:
issues = [
    "The projector in Block A classroom is not working",
    "There is no drinking water in the hostel",
    "There is a medical emergency in the hostel",
    "The canteen food quality is very poor",
    "I need information about career opportunities",
    "The online learning portal is not accessible"
]

for issue in issues:

    # Unpack the tuple returned by analyze_grievance directly
    category, priority, score, department, action, reasons = analyze_grievance(issue)

    print("\nGrievance:", issue)
    print("Category:", category)
    print("Priority:", priority)
    print("Score:", score, "/ 10")
    print("Department:", department)
    print("Action:", action)
    print("-" * 80)

In [ ]:
import joblib

joblib.dump(category_model, "category_model.pkl")

print("Category model saved.")

In [ ]:
%%writefile app.py

import streamlit as st
import joblib
import pandas as pd
import os
import uuid

st.set_page_config(
    page_title="GrievAI",
    page_icon="🎓",
    layout="wide"
)

category_model = joblib.load("category_model.pkl")

DATA_FILE = "grievances.csv"

COLUMNS = [
    "Grievance ID",
    "Student",
    "Grievance",
    "Category",
    "Priority",
    "Priority Score",
    "Responsible Department",
    "Recommended Action",
    "Priority Reason",
    "Status"
]

department_map = {
    "Academic Support and Resources": "Academic Support Cell",
    "Food and Cantines": "Canteen and Food Services",
    "Financial Support": "Student Finance Cell",
    "Online learning": "Digital Learning Cell",
    "Career opportunities": "Career Development Cell",
    "International student experiences": "International Student Cell",
    "Athletics and sports": "Sports Department",
    "Housing and Transportation": "Hostel and Transport Cell",
    "Health and Well-being Support": "Health and Wellness Cell",
    "Activities and Travelling": "Student Activities Cell",
    "Student Affairs": "Student Affairs Cell",
    "Emergency and Safety": "Emergency and Safety Cell"
}

action_map = {
    "Academic Support and Resources": "Assign the grievance to academic or technical support staff.",
    "Food and Cantines": "Forward the grievance to the canteen and food services team.",
    "Financial Support": "Forward the grievance to the student finance cell.",
    "Online learning": "Assign the issue to the digital learning support team.",
    "Career opportunities": "Forward the request to the career development cell.",
    "International student experiences": "Forward the grievance to the international student support cell.",
    "Athletics and sports": "Forward the issue to the sports department.",
    "Housing and Transportation": "Forward the grievance to the hostel or transport team.",
    "Health and Well-being Support": "Forward the grievance to the health and wellness cell.",
    "Activities and Travelling": "Forward the request to the student activities cell.",
    "Student Affairs": "Forward the grievance to the student affairs cell.",
    "Emergency and Safety": "Immediately forward the grievance to the appropriate emergency and safety response team."
}


def recommend_action(category, priority, grievance_text, department):
    text = grievance_text.lower()

    emergency_words = [
        "fire", "accident", "violence", "unsafe",
        "emergency", "danger", "medical emergency"
    ]

    if any(word in text for word in emergency_words):
        return "Immediately escalate to the Emergency and Safety Cell for urgent assessment."

    action = action_map.get(
        category,
        f"Forward the grievance to {department} for manual review."
    )

    if priority in ["Critical", "High"]:
        action += " Give this grievance priority and follow up promptly."
    elif priority == "Medium":
        action += " Track the case and follow up through the normal resolution process."

    return action


def calculate_priority(text):
    text = text.lower()

    critical_words = [
        "emergency", "fire", "accident", "danger",
        "unsafe", "violence", "medical emergency"
    ]

    high_words = [
        "urgent", "not working", "broken", "no water",
        "water problem", "water supply", "water shortage",
        "water issue", "water is not coming",
        "water not coming", "water is not working",
        "water not working", "water unavailable",
        "water not available", "power failure",
        "electricity", "security", "food poisoning",
        "severe", "stopped working"
    ]

    medium_words = [
        "delay", "problem", "issue",
        "difficulty", "unable", "poor"
    ]

    low_words = [
        "information", "query", "suggestion",
        "feedback", "enquiry", "career guidance",
        "guidance"
    ]

    if any(word in text for word in critical_words):
        return "Critical", 10, ["Critical safety or emergency indicator"]

    if any(word in text for word in high_words):
        return "High", 4, ["High urgency or service disruption"]

    if any(word in text for word in medium_words):
        return "Medium", 2, ["General service issue"]

    if any(word in text for word in low_words):
        return "Low", 1, ["Information or feedback request"]

    return "Low", 0, ["No strong urgency indicators detected"]


def analyze_grievance(issue):
    text = issue.lower()

    emergency_words = [
        "fire", "emergency", "accident",
        "danger", "unsafe", "violence",
        "medical emergency"
    ]

    if any(word in text for word in emergency_words):
        category = "Emergency and Safety"

    elif "water" in text:
        category = "Housing and Transportation"

    elif (
        "projector" in text
        or "classroom" in text
        or "academic resource" in text
        or "lecture" in text
    ):
        category = "Academic Support and Resources"

    else:
        category = category_model.predict([issue])[0]

    priority, score, reasons = calculate_priority(issue)

    if category == "Housing and Transportation" and "water" in text:
        if score < 4:
            priority = "High"
            score = 4
            reasons = ["High urgency or service disruption"]

    department = department_map.get(
        category,
        "Student Grievance Cell"
    )

    action = recommend_action(
        category,
        priority,
        issue,
        department
    )

    return category, priority, score, department, action, reasons


def load_grievances():
    if not os.path.exists(DATA_FILE):
        return pd.DataFrame(columns=COLUMNS)

    df = pd.read_csv(DATA_FILE).fillna("")

    if "Student" not in df.columns:
        df["Student"] = ""

    if "Student Name" in df.columns:
        df["Student"] = df["Student"].where(
            df["Student"].astype(str).str.strip() != "",
            df["Student Name"]
        )

    if "Responsible Department" not in df.columns:
        df["Responsible Department"] = ""

    if "Department" in df.columns:
        df["Responsible Department"] = (
            df["Responsible Department"].where(
                df["Responsible Department"].astype(str).str.strip() != "",
                df["Department"]
            )
        )

    for column in COLUMNS:
        if column not in df.columns:
            df[column] = ""

    return df[COLUMNS]


def save_grievances(df):
    df[COLUMNS].to_csv(DATA_FILE, index=False)


st.title("🎓 GrievAI")
st.subheader("AI-Powered University Grievance Analysis and Resolution Support")

mode = st.sidebar.radio(
    "Select Portal",
    ["Student Portal", "Manager Dashboard"]
)

if mode == "Student Portal":
    st.header("👩‍🎓 Student Portal")

    option = st.radio(
        "Choose an option",
        ["Submit Grievance", "Check Grievance Status"]
    )

    if option == "Submit Grievance":
        st.write(
            "Submit your university grievance. "
            "Your grievance ID will be generated after submission."
        )

        name = st.text_input("Student Name")
        issue = st.text_area("Describe your grievance")

        if st.button("Submit Grievance"):
            if not name.strip() or not issue.strip():
                st.warning("Please enter your name and grievance.")

            else:
                category, priority, score, department, action, reasons = (
                    analyze_grievance(issue)
                )

                grievance_id = "GRV-" + uuid.uuid4().hex[:8].upper()

                grievances_df = load_grievances()

                new_record = {
                    "Grievance ID": grievance_id,
                    "Student": name.strip(),
                    "Grievance": issue.strip(),
                    "Category": category,
                    "Priority": priority,
                    "Priority Score": score,
                    "Responsible Department": department,
                    "Recommended Action": action,
                    "Priority Reason": "; ".join(reasons),
                    "Status": "Submitted"
                }

                grievances_df = pd.concat(
                    [grievances_df, pd.DataFrame([new_record])],
                    ignore_index=True
                )

                save_grievances(grievances_df)

                st.success("Your grievance has been submitted successfully.")
                st.write("Save your grievance ID to check the status later.")
                st.code(grievance_id, language=None)

                st.caption(
                    "Use the copy icon beside the grievance ID to copy it."
                )

    else:
        st.subheader("Check Grievance Status")

        grievance_id = st.text_input("Enter your Grievance ID")

        if st.button("Check Status"):
            if not grievance_id.strip():
                st.warning("Please enter your grievance ID.")

            else:
                grievances_df = load_grievances()

                matches = grievances_df[
                    grievances_df["Grievance ID"].astype(str).str.upper()
                    == grievance_id.strip().upper()
                ]

                if matches.empty:
                    st.error("No grievance found with that ID.")

                else:
                    record = matches.iloc[-1]

                    st.write("**Grievance ID:**", record["Grievance ID"])
                    st.write("**Grievance:**", record["Grievance"])
                    st.write("**Status:**", record["Status"])

else:
    st.header("🧑‍💼 Manager Dashboard")

    grievances_df = load_grievances()

    total_count = len(grievances_df)

    critical_count = (
        (grievances_df["Priority"] == "Critical").sum()
        if total_count else 0
    )

    high_count = (
        (grievances_df["Priority"] == "High").sum()
        if total_count else 0
    )

    medium_count = (
        (grievances_df["Priority"] == "Medium").sum()
        if total_count else 0
    )

    resolved_count = (
        (grievances_df["Status"] == "Resolved").sum()
        if total_count else 0
    )

    col1, col2, col3, col4, col5 = st.columns(5)

    col1.metric("Total Grievances", total_count)
    col2.metric("Critical", int(critical_count))
    col3.metric("High", int(high_count))
    col4.metric("Medium", int(medium_count))
    col5.metric("Resolved", int(resolved_count))

    st.subheader("All Grievances")

    if grievances_df.empty:
        st.info("No grievances have been submitted yet.")

    else:
        st.dataframe(
            grievances_df,
            use_container_width=True,
            hide_index=True
        )

        st.subheader("Update Grievance Status")

        selected_id = st.selectbox(
            "Select Grievance ID",
            grievances_df["Grievance ID"].astype(str).tolist()
        )

        selected_record = grievances_df[
            grievances_df["Grievance ID"].astype(str) == selected_id
        ].iloc[0]

        st.write("**Student:**", selected_record["Student"])
        st.write("**Grievance:**", selected_record["Grievance"])
        st.write("**Category:**", selected_record["Category"])
        st.write("**Priority:**", selected_record["Priority"])
        st.write(
            "**Responsible Department:**",
            selected_record["Responsible Department"]
        )
        st.write(
            "**Recommended Action:**",
            selected_record["Recommended Action"]
        )

        status_options = ["Submitted", "In Progress", "Resolved"]
        current_status = selected_record["Status"]

        if current_status not in status_options:
            current_status = "Submitted"

        new_status = st.selectbox(
            "New Status",
            status_options,
            index=status_options.index(current_status)
        )

        if st.button("Update Status"):
            grievances_df.loc[
                grievances_df["Grievance ID"].astype(str) == selected_id,
                "Status"
            ] = new_status

            save_grievances(grievances_df)

            st.success("Grievance status updated successfully.")
            st.rerun()

In [ ]:
from app import analyze_grievance

print(analyze_grievance("the water is not coming in I block"))

In [ ]:
!streamlit run app.py > streamlit.log 2>&1 &

In [ ]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8501)")
print(url)

In [ ]:
!ps aux | grep streamlit

In [ ]:
!cat streamlit.log

In [ ]:
!pkill -f "streamlit run app.py"

In [ ]:
!streamlit run app.py --server.address=0.0.0.0 --server.port=8501 --server.enableXsrfProtection=false --server.enableCORS=false > streamlit.log 2>&1 &

In [ ]:
!cat streamlit.log

In [ ]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8501)")
print(url)

In [ ]:
!pkill -f "streamlit run app.py"

In [ ]:
!streamlit run app.py --server.address=0.0.0.0 --server.port=8501 --server.enableXsrfProtection=false --server.enableCORS=false > streamlit.log 2>&1 &

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

predictions = category_model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Category Classification Accuracy:", round(accuracy * 100, 2), "%")
print()
print(classification_report(y_test, predictions, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    xticks_rotation=90
)

plt.title("GrievAI Category Classification Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
print("Total records:", len(df))
print("Number of categories:", df["Genre"].nunique())

print("\nCategory distribution:")
print(df["Genre"].value_counts())

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate reports:")
print(df["Reports"].duplicated().sum())

In [ ]:
test_cases = [
    "The projector in Block A classroom is not working.",
    "The food served in the college canteen is very poor.",
    "I am unable to access the online learning platform.",
    "I need financial assistance for my education fees.",
    "There is a fire emergency in the hostel and students are in danger.",
    "I need guidance regarding career opportunities.",
    "There is a water problem in the hostel.",
    "I have difficulty accessing academic resources."
]

for i, issue in enumerate(test_cases, 1):

    category, priority, score, department, action, reasons = analyze_grievance(issue)

    print("Test Case:", i)
    print("Grievance:", issue)
    print("Category:", category)
    print("Priority:", priority)
    print("Score:", score, "/ 10")
    print("Department:", department)
    print("-" * 70)

In [ ]:
import pandas as pd
import os

file_path = "grievances.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("Columns found in grievances database:")
    print(df.columns.tolist())
    print("\nLatest 5 submissions:")
    print(df.tail(5).to_string(index=False))
else:
    print(f"The file '{file_path}' does not exist yet.")
    print("It will be automatically created once you submit a grievance through the Streamlit App web interface.")

In [ ]:
from app import analyze_grievance

tests = [
    "water is not coming in I block",
    "the water in the a block is not working",
    "The water is off in the I block",
    "The projector in the I block room 208 is not working"
]

for issue in tests:
    print("\nGrievance:", issue)
    print(analyze_grievance(issue))

In [ ]:
import sys
import platform
import sklearn
import pandas
import streamlit
import joblib

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pandas.__version__)
print("Streamlit:", streamlit.__version__)
print("Joblib:", joblib.__version__)

try:
    import psutil
    print("RAM (GB):", round(psutil.virtual_memory().total / (1024**3), 2))
except ImportError:
    print("RAM: unavailable")

try:
    import subprocess
    print(subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True
    ).stdout or "No NVIDIA GPU detected")
except Exception:
    print("GPU details unavailable")